# Inference System – Austrian Tax Law

**Model:** GPT-5.4-mini (OpenAI API)

---

## How does the model work?

This notebook uses inference-only mode, meaning the model is used directly
without any additional training or retrieval. Each question from the dataset
is sent to the OpenAI API and answered based solely on the model's pretrained
knowledge.

---

## Setup

1. Upload dataset_clean.csv to the Colab environment
2. Enter your OpenAI API key when prompted in Cell 6
3. Run all cells in order

---

## Data

The dataset contains 643 questions on Austrian tax law. Each question is identified by
a unique id and contains a prompt in German. The dataset was used exclusively
as a test set.

# Cell 1 – Load dataset
The dataset_clean with the 643 questions is loaded from CSV into a pandas DataFrame. Each question is identified by a unique id and a prompt

In [ ]:
import pandas as pd

df = pd.read_csv("/content/dataset_clean.csv")

df.head()

,id,prompt
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ..."
2,CORP-TAX-003,"Welche Körperschaften sind verpflichtet, sämtl..."
3,CORP-TAX-004,Wie wird eine Dividende einer österreichischen...
4,CORP-TAX-005,Was unterscheidet die steuerliche Behandlung e...


# Cell 3 – Sample
A reproducible random sample of 10 questions is drawn for testing purposes.
random_state=42 ensures the same sample is selected every time.

In [ ]:
sample_df = df.sample(10, random_state=42)

# Cell 3 – Inspect dataset
Quick check of the dataset structure: column names and total number of rows.

In [ ]:
print(df.columns)
print(len(df))

Index(['id', 'prompt'], dtype='object')
643


# Cell 4 – Install OpenAI
Installation of the official OpenAI Python client library.

In [ ]:
!pip install -q openai

# Cell 5 – API key
The OpenAI API key is entered securely via a password prompt and stored as
an environment variable. The key is never hardcoded in the notebook.

In [ ]:
import os
from getpass import getpass



os.environ["OPENAI_API_KEY"] = getpass("Enter your API key: ")

Enter your API key: ··········


# Cell 6 – Inference function
Defines the answer_question function which sends each question to the
GPT model via the OpenAI Chat Completions API. The prompt instructs the
model to answer concisely in plain text and cite the relevant legal basis
where possible. temperature=0 ensures deterministic outputs.

In [ ]:
from openai import OpenAI

client = OpenAI()

def answer_question(question):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": f"""Beantworte die folgende steuerrechtliche Frage für Österreich kurz und präzise.
Gib nur die Antwort, keine Einleitung. Reiner Text: kein Fettgeschriebenes etc..
Wenn möglich, nenne die relevante gesetzliche Grundlage.
Question:
{question}"""
            }
        ]
    )
    return response.choices[0].message.content.strip()

# Cell 7 – Single question test
Tests the inference function on the first question in the dataset to verify
the API connection and prompt format before running the full dataset.

In [ ]:
test_q = df.loc[0, "prompt"]
print(test_q)
print()
print(answer_question(test_q))

Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?

Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer ist das zu versteuernde Einkommen der Körperschaft, also der nach den Vorschriften des EStG/KStG ermittelte Gesamtbetrag der Einkünfte abzüglich Sonderausgaben, Freibeträge und sonstiger Abzüge. Rechtsgrundlage: § 7 KStG 1988.


#Cell 8 – Sample run
Runs the inference function on the 10-question sample. Results are collected
as a list of id and answer pairs. Errors are caught and stored as "ERROR"
to avoid interrupting the loop.

In [ ]:
sample_df = df.sample(10, random_state=42)

sample_answers = []

for i, row in sample_df.iterrows():
    qid = row["id"]
    question = row["prompt"]

    try:
        answer = answer_question(question)
    except Exception as e:
        answer = "ERROR"

    sample_answers.append({
        "id": qid,
        "answer": answer
    })

    print(f"Done: {qid}")

Done: SELF-070
Done: VAT-INTL-047
Done: EMP-045
Done: GRESt-AT-051
Done: TAX-INTL-013
Done: ESTG27-034
Done: SELF-036
Done: VAT-INTL-063
Done: SELF-051
Done: VAT-DOM-015


# Cell 9 – Save sample output
The sample results are saved as a CSV file and displayed for inspection.

In [ ]:
sample_output_df = pd.DataFrame(sample_answers)
sample_output_df.to_csv("/content/sample_inference_output.csv", index=False)
sample_output_df.head(10)

,id,answer
0,SELF-070,Eine Sicherheitseinrichtung für Registrierkass...
1,VAT-INTL-047,Bei einer sonstigen Leistung an einen Unterneh...
2,EMP-045,Nein. Ein Klimaticket ist zwar grundsätzlich s...
3,GRESt-AT-051,Die Selbstberechnung der Grunderwerbsteuer dar...
4,TAX-INTL-013,Ja. Für die österreichische Besteuerung ist gr...
5,ESTG27-034,Die spätere Überführung der Futures in ein Dep...
6,SELF-036,Das Kommunalsteuergesetz regelt die Kommunalst...
7,VAT-INTL-063,Bei einer sonstigen Leistung an eine Privatper...
8,SELF-051,Pflichtversichert nach dem GSVG sind vor allem...
9,VAT-DOM-015,Nein. Bei der kurzfristigen Vermietung eines S...


# Cell 10 – Full run (all 643 questions)
The inference function is applied to all 643 questions in the dataset.
Results are saved as inference_output.csv.

In [ ]:
answers = []

for i, row in df.iterrows():
    qid = row["id"]
    question = row["prompt"]

    try:
        answer = answer_question(question)
    except Exception as e:
        answer = "ERROR"

    answers.append({
        "id": qid,
        "answer": answer
    })

    if (i + 1) % 20 == 0:
        print(f"{i+1}/{len(df)} done")

20/643 done
40/643 done
60/643 done
80/643 done
100/643 done
120/643 done
140/643 done
160/643 done
180/643 done
200/643 done
220/643 done
240/643 done
260/643 done
280/643 done
300/643 done
320/643 done
340/643 done
360/643 done
380/643 done
400/643 done
420/643 done
440/643 done
460/643 done
480/643 done
500/643 done
520/643 done
540/643 done
560/643 done
580/643 done
600/643 done
620/643 done
640/643 done


# Cell 11 – Download output
The completed CSV file is downloaded from the Colab environment to the
local machine for submission.

In [ ]:
output_df = pd.DataFrame(answers)
output_df.to_csv("/content/inference_output.csv", index=False)
output_df.head()

,id,answer
0,CORP-TAX-001,Die steuerliche Bemessungsgrundlage für die Kö...
1,CORP-TAX-002,Ein zinsloses Darlehen an den Gesellschafter f...
2,CORP-TAX-003,Körperschaften öffentlichen Rechts sind verpfl...
3,CORP-TAX-004,(a) Bei der österreichischen Tochtergesellscha...
4,CORP-TAX-005,Eine offene Ausschüttung ist gesellschaftsrech...


In [ ]:
from google.colab import files
files.download("/content/inference_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>